In [1]:
# Tiny Shakespeare — Text Generation

## Objective

# Load the trained MiniGPT model and generate new text character-by-character.

### Generation pipeline

# 1. Load the trained model.
# 2. Load the vocabulary and tokenizer mappings.
# 3. Encode a starting prompt.
# 4. Predict the next character.
# 5. Sample from the probability distribution.
# 6. Append the predicted character.
# 7. Repeat until the requested length is reached.

# This is autoregressive text generation.

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
print("PyTorch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

PyTorch: 2.14.0
MPS available: True


In [3]:
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print("Using device:", device)

Using device: mps


In [4]:
vocab_size = 63
context_length = 128

embedding_dim = 256
num_heads = 4
num_layers = 4

dropout = 0.2

In [5]:
class Head(nn.Module):

    def __init__(
        self,
        embedding_dim,
        head_dim,
        context_length,
        dropout
    ):
        super().__init__()

        self.key = nn.Linear(
            embedding_dim,
            head_dim,
            bias=False
        )

        self.query = nn.Linear(
            embedding_dim,
            head_dim,
            bias=False
        )

        self.value = nn.Linear(
            embedding_dim,
            head_dim,
            bias=False
        )

        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "tril",
            torch.tril(
                torch.ones(
                    context_length,
                    context_length
                )
            )
        )

    def forward(self, x):

        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        attention_scores = q @ k.transpose(-2, -1)

        attention_scores = (
            attention_scores
            * (k.shape[-1] ** -0.5)
        )

        attention_scores = attention_scores.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        attention_weights = F.softmax(
            attention_scores,
            dim=-1
        )

        attention_weights = self.dropout(
            attention_weights
        )

        v = self.value(x)

        return attention_weights @ v

In [6]:
class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        embedding_dim,
        num_heads,
        context_length,
        dropout
    ):
        super().__init__()

        head_dim = embedding_dim // num_heads

        self.heads = nn.ModuleList([
            Head(
                embedding_dim,
                head_dim,
                context_length,
                dropout
            )
            for _ in range(num_heads)
        ])

        self.projection = nn.Linear(
            embedding_dim,
            embedding_dim
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        outputs = [
            head(x)
            for head in self.heads
        ]

        x = torch.cat(
            outputs,
            dim=-1
        )

        x = self.projection(x)

        return self.dropout(x)

In [7]:
class FeedForward(nn.Module):

    def __init__(
        self,
        embedding_dim,
        dropout
    ):
        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                embedding_dim,
                4 * embedding_dim
            ),

            nn.GELU(),

            nn.Linear(
                4 * embedding_dim,
                embedding_dim
            ),

            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.network(x)

In [8]:
class TransformerBlock(nn.Module):

    def __init__(
        self,
        embedding_dim,
        num_heads,
        context_length,
        dropout
    ):
        super().__init__()

        self.ln1 = nn.LayerNorm(
            embedding_dim
        )

        self.attention = MultiHeadAttention(
            embedding_dim,
            num_heads,
            context_length,
            dropout
        )

        self.ln2 = nn.LayerNorm(
            embedding_dim
        )

        self.ffn = FeedForward(
            embedding_dim,
            dropout
        )

    def forward(self, x):

        x = x + self.attention(
            self.ln1(x)
        )

        x = x + self.ffn(
            self.ln2(x)
        )

        return x

In [9]:
class MiniGPT(nn.Module):

    def __init__(
        self,
        vocab_size,
        context_length,
        embedding_dim,
        num_heads,
        num_layers,
        dropout
    ):
        super().__init__()

        self.context_length = context_length

        self.token_embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.position_embedding = nn.Embedding(
            context_length,
            embedding_dim
        )

        self.blocks = nn.Sequential(
            *[
                TransformerBlock(
                    embedding_dim,
                    num_heads,
                    context_length,
                    dropout
                )
                for _ in range(num_layers)
            ]
        )

        self.ln_final = nn.LayerNorm(
            embedding_dim
        )

        self.lm_head = nn.Linear(
            embedding_dim,
            vocab_size
        )

    def forward(self, idx):

        B, T = idx.shape

        token_embeddings = self.token_embedding(idx)

        positions = torch.arange(
            T,
            device=idx.device
        )

        position_embeddings = self.position_embedding(
            positions
        )

        x = token_embeddings + position_embeddings

        x = self.blocks(x)

        x = self.ln_final(x)

        logits = self.lm_head(x)

        return logits

In [10]:
MODEL_PATH = "../models/minigpt_checkpoint.pt"

print("Model path:", MODEL_PATH)
print("Exists:", os.path.exists(MODEL_PATH))

Model path: ../models/minigpt_checkpoint.pt
Exists: True


In [11]:
checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
    weights_only=False
)

print("Checkpoint loaded.")
print("Training iteration:", checkpoint["iteration"])
print("Training loss:", checkpoint["train_loss"])
print("Validation loss:", checkpoint["val_loss"])

Checkpoint loaded.
Training iteration: 5000
Training loss: 1.3438644409179688
Validation loss: 1.4844762086868286


In [12]:
vocab = checkpoint["vocab"]
stoi = checkpoint["stoi"]
itos = checkpoint["itos"]

print("Vocabulary size:", len(vocab))
print("Vocabulary:", vocab)

Vocabulary size: 63
Vocabulary: ['\n', ' ', '!', '&', "'", ',', '-', '.', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [13]:
model = MiniGPT(
    vocab_size=vocab_size,
    context_length=context_length,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    dropout=dropout
).to(device)

In [14]:
model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Trained model loaded.")

Trained model loaded.


In [15]:
def encode(text):
    return [
        stoi[character]
        for character in text
    ]


def decode(tokens):
    return "".join(
        itos[token]
        for token in tokens
    )

In [16]:
@torch.no_grad()
def generate(
    model,
    prompt,
    max_new_tokens=300,
    temperature=1.0
):

    model.eval()

    tokens = encode(prompt)

    idx = torch.tensor(
        [tokens],
        dtype=torch.long,
        device=device
    )

    for _ in range(max_new_tokens):

        # Keep only the most recent context
        idx_cond = idx[:, -context_length:]

        # Model prediction
        logits = model(idx_cond)

        # Get prediction for final position
        logits = logits[:, -1, :]

        # Temperature
        logits = logits / temperature

        # Convert to probabilities
        probabilities = F.softmax(
            logits,
            dim=-1
        )

        # Sample next character
        next_token = torch.multinomial(
            probabilities,
            num_samples=1
        )

        # Append token
        idx = torch.cat(
            (idx, next_token),
            dim=1
        )

    return decode(
        idx[0].tolist()
    )

In [17]:
prompt = "ROMEO:"

generated_text = generate(
    model,
    prompt,
    max_new_tokens=300,
    temperature=0.8
)

print(generated_text)

ROMEO:
What, let her is these grace? then, and mark their sons times,
I'll regal take me to each tears to the last:
I shall not hear to speak your honour words,
By something of when of your strawful spring is people;
And which I must and him such can see, he would
whose thy doubted lords; then from his fa


In [18]:
prompts = [
    "ROMEO:",
    "JULIET:",
    "KING:",
    "HAMLET:",
    "To be",
]

for prompt in prompts:

    print("=" * 70)
    print("PROMPT:", prompt)
    print("=" * 70)

    output = generate(
        model,
        prompt,
        max_new_tokens=200,
        temperature=0.8
    )

    print(output)
    print()

PROMPT: ROMEO:
ROMEO:
My lords; and where being the daughter constant of shall reber
In the other stand of the might royal dangerous
That was sounds must come and serves;
And I will not we reason myself and I
I turre to s

PROMPT: JULIET:
JULIET:
My fair, sir, and commend and so, say, he sleep;
He shall you shalt see as yet shall be and wine
Than we do your son all thing most surved would be
your honour country's enserent of here shall
do not

PROMPT: KING:
KING: and here comes once we to but not,
That have good a power to the despect
Of ever many times and air repent.

GLOUCESTER:
My lord shall souls paint thy doom.

FLORIZEL:
Do which you have done and my h

PROMPT: HAMLET:
HAMLET:
What, she is much what I should this warlike
The days dear which the land was finds!

COMINIUS:
Now, then what dost me the sun of me?

CORIOLANUS:
Then I have I do not up, he
proud more better enough

PROMPT: To be
To be the daughter of your grace and fair,
Who see of Lancaster her to God me in me,
But som

In [19]:
prompt = "ROMEO:"

temperatures = [0.3, 0.8, 1.2]

for temperature in temperatures:

    print("=" * 70)
    print(f"TEMPERATURE: {temperature}")
    print("=" * 70)

    output = generate(
        model,
        prompt,
        max_new_tokens=250,
        temperature=temperature
    )

    print(output)
    print()

TEMPERATURE: 0.3
ROMEO:
Should be the season, that I have been prove thee,
The sun stand the prince of the world,
The people more than the man the state of the prince
That thou art the senator of the body is soul,
That have should be run the sent of the rest,
Who shall I s

TEMPERATURE: 0.8
ROMEO:
For thy poor man.

MENENIUS:
Poor thy dream?

KING RICHARD II:
Who best, what we had, do sir, none it, fair bad the
reating of my swalloy's land Plintague,
Come to his counted here, had been his poison

LADY GREY:
Which, my I range than their soul d

TEMPERATURE: 1.2
ROMEO:
Thas one what was
I'll see this tendue him. The made under in the tale? what thou!

MENENIUS:
They? I would, your commendings,
Cudge King God Edwards and shall be all child.

NORFOLK:
Sduct how with our Criple's refut's horse woe sleep,
Letteo and f



In [20]:
output = generate(
    model,
    "ROMEO:",
    max_new_tokens=800,
    temperature=0.8
)

print(output)

ROMEO:
What, sayst you pointed this? I shall not to see make thee?

CORIOLANUS:
I am I have rest come this privation stoo end.

HERMIONE:

GLOUCESTER:

LADY CABERLAND:
O prayers and love away.

LEONTES:

AUFIDIUS:
See, my lord, think, and see them I live to know.

LUCIO:
Sir, and to answer?

LUCENTIO:
Sir, I cannot for his knows heart we shall remembrace than he shouldst this:
O, thou art that if thou talk'd their daughter.

CORIOLANUS:
And, noble to know us it is.

Second Murderer:
I tell me, but which I have to been your people,
And he draw you should speech your head.

DUKE VINCENTIO:
Come, sir, thou half use to be devide thy gods,
Take them by mind; burns, but this, who
Have day I will not not for the suspice we have in a stifled,
Which she is behalf with him from it.
Thou revenge back the k


In [21]:
def top_k_sample(logits, k=10):

    values, indices = torch.topk(
        logits,
        k
    )

    probabilities = F.softmax(
        values,
        dim=-1
    )

    selected = torch.multinomial(
        probabilities,
        num_samples=1
    )

    return indices.gather(
        -1,
        selected
    )

In [22]:
@torch.no_grad()
def generate(
    model,
    prompt,
    max_new_tokens=500,
    temperature=0.8,
    top_k=10
):

    model.eval()

    tokens = encode(prompt)

    idx = torch.tensor(
        [tokens],
        dtype=torch.long,
        device=device
    )

    for _ in range(max_new_tokens):

        idx_cond = idx[:, -context_length:]

        logits = model(idx_cond)

        logits = logits[:, -1, :]

        logits = logits / temperature

        # Top-K filtering
        values, indices = torch.topk(
            logits,
            top_k
        )

        probabilities = F.softmax(
            values,
            dim=-1
        )

        selected = torch.multinomial(
            probabilities,
            num_samples=1
        )

        next_token = indices.gather(
            -1,
            selected
        )

        idx = torch.cat(
            (idx, next_token),
            dim=1
        )

    return decode(
        idx[0].tolist()
    )

In [23]:
output = generate(
    model,
    "ROMEO:",
    max_new_tokens=800,
    temperature=0.8,
    top_k=10
)

print(output)

ROMEO:
Be must do reason our fair than our sensel
As maid the while world to the mother worth,
That is be dangerous shall be to the faults,
And who shall thy soul's winds. If thou art thy body day'
Thy part of thing and first and mother, he will be
Thy swords thou art a do to his crown fortune.

CAMILLO:
Come, my lords; and struly brother in his sight.

ROMEO:
It is thy sun ill see as hear all die,
Believe us the pirity of the read of the sick,
That thou sun this signity to any holy here
That shall the severeign so to his service,
But he dear the mind be the pause of me.

LEONTES:
Then, I cannot it shall himself a creature
We are to such a first we have time to all.

LADY ANNE:
I am beat the secution.

LORD ROSS:
How faith thy father's prince! how all me, thou art moved,
For to power and bring t


In [24]:
output = generate(
    model,
    "ROMEO:",
    max_new_tokens=800,
    temperature=0.8,
    top_k=20
)

print(output)

ROMEO:
Must and make him and must should with seam of hell.

VALERIA:
Now, let the wars of my father: he do beseech me,
That now they not me are for other.

First Servingman:
Sir, the rest pity services, for whom you should be with
That say thy world charged pass'd him, if yet should you
Thou have givesty by how our good exile.

SICINIUS:
Why, thou would not of your steps?

CORIOLANUS:
Thou, then is welcome these
Of the secret, if the thrust didst did to the king,
And they had thinkear of safety their will,
Like the bed and tears as my come;
And so for the hundred her, fair to the city.

RICHARD:
My Lord Hastings of Norfolk, do not the mercy proad
As manner bound, then thou have done.

GLOUCESTER:
For it then you, no, for the slain's bones,
So the signiors hath been the thribed of the head,
Are 


In [25]:
prompt = "ROMEO:"

for k in [3, 5, 10, 20]:

    print("=" * 80)
    print(f"TOP-K = {k}")
    print("=" * 80)

    output = generate(
        model,
        prompt,
        max_new_tokens=300,
        temperature=0.8,
        top_k=k
    )

    print(output)
    print()

TOP-K = 3
ROMEO:
I am such most that tell thee, when I was as many son
That we was to me, a shorts of their storm,
And see him and have so see her shall been him,
Who is the change of these shouldst have he was a desper
To send his part of this sound that have
To the subject of the people and his sister,
Which they

TOP-K = 5
ROMEO:
Have now made the comported of the cause of
The war of the course is the with our suit
And me with the true to stay.

LEONTES:
I will strong it; but in the tribunes.

BENVOLIO:
Then, then, though a cur children in the sin,
And holy words and time to be this tongue in his part
What she shall that sh

TOP-K = 10
ROMEO:
What, and I desperate, my grow lord?
If thou shalt and these stempts, and I
When deeds are my by heart, and now not man
To bad the world have a throner of the state:
Therefore I think hath me said but his soon.

KING RICHARD III:
Why, a shall I will should stand she would shall I take to
contenced 

TOP-K = 20
ROMEO:
Therefore I have done when 